# BiomedCLIP Baseline for PAR-VAE Comparison

This notebook extracts BiomedCLIP embeddings from CT slices and evaluates them
using the **identical probe + evaluation protocol** as PAR-VAE:
- Same train/val/test splits (volume-level patient splitting)
- Same downstream probes (LogReg for S1/S2, RBF-SVM for S3)
- Same multi-seed validation (seeds 16, 42, 999)
- Same metrics: AUC, F1, Accuracy, Miss Rate at S3

**Preprocessing note (for paper):**
CT slices (single-channel, [-1,1]) are repeated to 3 channels and
renormalized to BiomedCLIP's expected ImageNet stats before encoding.
This is the standard adaptation for grayscale medical images with CLIP-based models.
No other modifications are made to the BiomedCLIP encoder.

In [1]:
# ── Install dependencies ──────────────────────────────────────────────────────
# BiomedCLIP is distributed via open_clip
!pip install -q open_clip_torch huggingface_hub scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.

In [2]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import open_clip
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, confusion_matrix
)
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Seeds to match PAR-VAE evaluation exactly
SEEDS = [16, 42, 999]

Device: cuda


## 1. Reproducibility

In [3]:
def set_seed(seed: int):
    """Mirror the set_seed() from attrisivae-model.ipynb exactly."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 2. Dataset — CT slices reformatted for BiomedCLIP

BiomedCLIP's vision encoder (ViT-B/16) expects:
- Input shape: `(B, 3, 224, 224)`
- Normalisation: ImageNet mean/std `([0.481, 0.457, 0.408], [0.268, 0.261, 0.275])`

Your CT slices are `(1, 512, 512)` in `[-1, 1]`.
The adapter below handles the conversion without touching your existing pipeline.

In [4]:
import torchvision.transforms as T

# BiomedCLIP's exact normalisation stats
BIOMEDCLIP_MEAN = (0.48145466, 0.4578275,  0.40821073)
BIOMEDCLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

BIOMEDCLIP_TRANSFORM = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.Normalize(mean=BIOMEDCLIP_MEAN, std=BIOMEDCLIP_STD),
])


class CTDataset_BiomedCLIP(Dataset):
    """
    Wraps the same CSV files used by CTDataset_ARSIVAE.
    Returns CT slices pre-processed for BiomedCLIP's ViT-B/16 encoder.

    Conversion pipeline per slice:
      1. Load .npy  →  float32 array in [-1, 1]   (your existing format)
      2. Rescale [-1,1] → [0,1]  (neutral window; avoids negative values
         before converting to pseudo-RGB)
      3. Repeat single channel → 3 identical channels  (pseudo-RGB)
      4. Resize 512→224, CenterCrop, ImageNet normalise

    The label and id fields are kept identical to CTDataset_ARSIVAE
    so the same downstream evaluation code works unchanged.
    """

    def __init__(self, csv_path=None, df=None):
        if df is not None:
            self.df = df.reset_index(drop=True)
        elif csv_path is not None:
            self.df = pd.read_csv(csv_path)
        else:
            raise ValueError("Provide csv_path or df.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── load slice (your existing format: float32, shape (512,512), range [-1,1])
        ct = np.load(row['ct_path']).astype(np.float32)   # (H, W)

        # ── step 1: [-1,1] → [0,1]
        ct_01 = (ct + 1.0) / 2.0                          # (H, W)

        # ── step 2: (H,W) → (1,H,W) tensor, then repeat to (3,H,W)
        ct_tensor = torch.from_numpy(ct_01).unsqueeze(0)  # (1, H, W)
        ct_rgb    = ct_tensor.repeat(3, 1, 1)             # (3, H, W)

        # ── step 3: resize / crop / normalise for BiomedCLIP
        ct_clip = BIOMEDCLIP_TRANSFORM(ct_rgb)             # (3, 224, 224)

        return {
            'ct_clip': ct_clip,
            'label':   torch.tensor(row['label'], dtype=torch.long),
            'id':      row['id'],
        }

## 3. Load BiomedCLIP (frozen encoder)

In [5]:
def load_biomedclip(device):
    """
    Load BiomedCLIP vision encoder from HuggingFace.
    Returns the encoder in eval mode with all parameters frozen.

    Output embedding dim: 512  (ViT-B/16 projection head)
    """
    model, _, _ = open_clip.create_model_and_transforms(
        'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
    )
    vision_encoder = model.visual   # ViT-B/16 + projection → 512-d
    vision_encoder = vision_encoder.to(device)
    vision_encoder.eval()

    # Freeze all parameters — we probe the representation, not fine-tune it
    for p in vision_encoder.parameters():
        p.requires_grad = False

    n_params = sum(p.numel() for p in vision_encoder.parameters())
    print(f'BiomedCLIP vision encoder loaded: {n_params/1e6:.1f}M params (frozen)')
    return vision_encoder


biomedclip_encoder = load_biomedclip(DEVICE)

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

BiomedCLIP vision encoder loaded: 86.2M params (frozen)


## 4. Feature extraction

Mirrors `extract_features()` from `attrisivae-model.ipynb` exactly —
same return dict shape so the same downstream code works.

In [6]:
def extract_biomedclip_features(encoder, loader, device):
    """
    Extract frozen BiomedCLIP embeddings for all slices in loader.

    Returns
    -------
    dict with keys:
        'latents'  : np.ndarray (N, 512)  — vision embeddings
        'labels'   : np.ndarray (N,)      — class labels
    """
    encoder.eval()
    latents, labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Extracting BiomedCLIP features'):
            x = batch['ct_clip'].to(device)       # (B, 3, 224, 224)
            emb = encoder(x)                       # (B, 512)

            # L2-normalise — standard practice for CLIP embeddings
            emb = emb / (emb.norm(dim=-1, keepdim=True) + 1e-8)

            latents.append(emb.cpu().numpy())
            labels.append(batch['label'].cpu().numpy())

    return {
        'latents': np.vstack(latents),
        'labels':  np.concatenate(labels),
    }

## 5. Downstream probe — identical to PAR-VAE evaluation

In [7]:
def run_probe(train_data, val_data, test_data, cohort, seed):
    """
    Train a downstream probe on BiomedCLIP embeddings.

    Probe choice mirrors PAR-VAE:
      S1, S2  → Logistic Regression
      S3      → RBF-SVM  (non-linear separability at high GGO burden)

    Parameters
    ----------
    train_data, val_data, test_data : dicts from extract_biomedclip_features()
    cohort : str  — 'S1', 'S2', or 'S3'
    seed   : int

    Returns
    -------
    dict of metrics
    """
    X_train, y_train = train_data['latents'], train_data['labels']
    X_val,   y_val   = val_data['latents'],   val_data['labels']
    X_test,  y_test  = test_data['latents'],  test_data['labels']

    # ── scale (fit on train only — no leakage)
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # ── probe selection
    if cohort in ('S1', 'S2'):
        probe = LogisticRegression(
            max_iter=1000, random_state=seed, C=1.0
        )
    else:  # S3
        probe = SVC(
            kernel='rbf', probability=True,
            random_state=seed, C=1.0, gamma='scale'
        )

    probe.fit(X_train, y_train)

    # ── val metrics (for reporting val–test gap)
    val_proba = probe.predict_proba(X_val)[:, 1]
    val_pred  = probe.predict(X_val)
    val_auc   = roc_auc_score(y_val, val_proba)
    val_acc   = accuracy_score(y_val, val_pred)

    # ── test metrics
    test_proba = probe.predict_proba(X_test)[:, 1]
    test_pred  = probe.predict(X_test)
    test_auc   = roc_auc_score(y_test, test_proba)
    test_acc   = accuracy_score(y_test, test_pred)
    test_f1    = f1_score(y_test, test_pred)

    # ── miss rate (false negatives on COVID class = label 1)
    # Matches the clinical safety metric in the PAR-VAE paper
    cm = confusion_matrix(y_test, test_pred)
    # cm layout: [[TN, FP], [FN, TP]]
    fn        = cm[1, 0] if cm.shape == (2, 2) else 0
    n_positive = (y_test == 1).sum()
    miss_rate  = fn / n_positive if n_positive > 0 else float('nan')

    return {
        'seed':       seed,
        'cohort':     cohort,
        'val_acc':    val_acc,
        'val_auc':    val_auc,
        'test_acc':   test_acc,
        'test_f1':    test_f1,
        'test_auc':   test_auc,
        'miss_rate':  miss_rate,
        'val_test_gap': val_auc - test_auc,
    }

## 6. Load data helper

Mirrors `load_frozen_dataset()` from `attrisivae-model.ipynb`.

In [8]:
# Cohort definitions — which severity labels to keep per cohort
# S1 = mild (CT-1) vs normal (CT-0)
# S2 = moderate (CT-2) vs normal (CT-0)
# S3 = severe (CT-3) vs normal (CT-0)
COHORT_SEVERITY = {'S1': 1, 'S2': 2, 'S3': 3}


def load_frozen_dataset(base_path, data_root):
    """
    Load the flat train/val/test CSVs from base_path.
    Returns full DataFrames — cohort filtering is done in make_cohort_loaders().
    Also fixes any path prefixes so ct_path points to the correct Kaggle location.
    """
    import os, pandas as pd

    train_df = pd.read_csv(os.path.join(base_path, 'train.csv'))
    val_df   = pd.read_csv(os.path.join(base_path, 'val.csv'))
    test_df  = pd.read_csv(os.path.join(base_path, 'test.csv'))

    # Fix paths: rewrite whatever prefix is in the CSV to data_root
    for df in [train_df, val_df, test_df]:
        for col in ['ct_path', 'mu_path', 'mask_path']:
            if col not in df.columns:
                continue
            # Strip everything up to and including 'ct_processed/' prefix variant,
            # then rebuild from data_root
            df[col] = df[col].apply(
                lambda p: os.path.join(
                    data_root,
                    'ct_processed',
                    os.path.basename(p)
                ) if pd.notna(p) else p
            )

    return train_df, val_df, test_df


def filter_cohort(df, cohort):
    """
    Filter a full split DataFrame to a binary cohort:
      - Keep rows where severity == 0 (normal, label=0)
      - Keep rows where severity == COHORT_SEVERITY[cohort] (covid, label=1)
    Reassigns 'label' to 0/1 regardless of what was in the CSV.
    """
    sev_col = 'severity' if 'severity' in df.columns else 'label'
    target_sev = COHORT_SEVERITY[cohort]

    normal = df[df[sev_col] == 0].copy()
    covid  = df[df[sev_col] == target_sev].copy()

    normal['label'] = 0
    covid['label']  = 1

    filtered = pd.concat([normal, covid], ignore_index=True)
    return filtered


def make_loaders(train_df, val_df, test_df, batch_size=64):
    """Build BiomedCLIP DataLoaders from already-filtered split DataFrames."""
    train_loader = DataLoader(
        CTDataset_BiomedCLIP(df=train_df),
        batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        CTDataset_BiomedCLIP(df=val_df),
        batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        CTDataset_BiomedCLIP(df=test_df),
        batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )
    return train_loader, val_loader, test_loader
    


## 7. Main evaluation loop

Run across all seeds and cohorts. Results are saved to CSV
in the same format as your PAR-VAE results for easy table merging.

In [9]:
def summarise(results_df):
    """
    Print mean ± std across seeds in the same format as Table 3
    in the PAR-VAE paper, so results can be directly inserted.
    """
    metrics = ['val_acc', 'test_acc', 'test_f1', 'test_auc', 'miss_rate', 'val_test_gap']
    print('\n' + '='*65)
    print(f'BiomedCLIP — Cohort {results_df["cohort"].iloc[0]} '
          f'(mean ± std, {len(results_df)} seeds)')
    print('='*65)
    for m in metrics:
        vals = results_df[m].values
        print(f'  {m:<18s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

    # Val–test AUC gap per seed (critical for shortcut learning diagnosis)
    print('\nPer-seed val–test AUC gap (shortcut learning indicator):')
    for _, row in results_df.iterrows():
        print(f"  Seed {row['seed']:3d}: val={row['val_auc']:.4f}  "
              f"test={row['test_auc']:.4f}  "
              f"gap={row['val_test_gap']:+.4f}")

## 8. Summary table (paper-ready)

## 9. Val–test gap diagnostic plot

Directly analogous to the CNN stability analysis in the paper.
A large, seed-varying gap = shortcut learning signal.

In [10]:
import matplotlib.pyplot as plt

def plot_seed_stability(results_df, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    seeds = results_df['seed'].values
    val_aucs  = results_df['val_auc'].values
    test_aucs = results_df['test_auc'].values
    miss_rates = results_df['miss_rate'].values

    # AUC per seed
    ax = axes[0]
    x = np.arange(len(seeds))
    ax.bar(x - 0.2, val_aucs,  0.35, label='Val AUC',  color='steelblue', alpha=0.8)
    ax.bar(x + 0.2, test_aucs, 0.35, label='Test AUC', color='tomato',    alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Seed {s}' for s in seeds])
    ax.set_ylabel('AUC')
    ax.set_title(f'BiomedCLIP — {COHORT} AUC per seed')
    ax.legend()
    ax.set_ylim(0.5, 1.05)
    ax.grid(axis='y', alpha=0.3)

    # Miss rate per seed
    ax = axes[1]
    bars = ax.bar(x, miss_rates * 100, color='darkorange', alpha=0.8)
    ax.axhline(1.6, color='green', linestyle='--',
               label='PAR-VAE S3 miss rate (1.6%)')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Seed {s}' for s in seeds])
    ax.set_ylabel('Miss Rate (%)')
    ax.set_title(f'BiomedCLIP — {COHORT} Miss Rate per seed')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    # Annotate bars with values
    for bar, val in zip(bars, miss_rates * 100):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


In [11]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
BASE_PATH  = '/kaggle/input/datasets/anshullmudyavar/processed-mosmed/processed'
DATA_ROOT  = '/kaggle/input/datasets/anshullmudyavar/processed-mosmed/processed'
BATCH_SIZE = 64
COHORTS    = ['S1', 'S2', 'S3']
# ─────────────────────────────────────────────────────────────────────────────

# Load full splits once — all severity levels in one go
print('Loading full dataset CSVs...')
full_train_df, full_val_df, full_test_df = load_frozen_dataset(BASE_PATH, DATA_ROOT)
print(f'  Train: {len(full_train_df):,} rows  |  '
      f'Val: {len(full_val_df):,}  |  Test: {len(full_test_df):,}')
print(f'  Severity counts in train: '
      f'{full_train_df["severity"].value_counts().sort_index().to_dict()}')

all_results = []

for COHORT in COHORTS:
    print(f'\n{"="*60}')
    print(f'BiomedCLIP Baseline — Cohort {COHORT} '
          f'(severity {COHORT_SEVERITY[COHORT]} vs 0)')
    print(f'{"="*60}')

    # ── Filter to this cohort's binary classification task ────────────────────
    train_df = filter_cohort(full_train_df, COHORT)
    val_df   = filter_cohort(full_val_df,   COHORT)
    test_df  = filter_cohort(full_test_df,  COHORT)

    print(f'  Train: {len(train_df):,}  '
          f'(normal={( train_df["label"]==0).sum()}  '
          f'covid={(train_df["label"]==1).sum()})')
    print(f'  Val  : {len(val_df):,}  '
          f'(normal={(val_df["label"]==0).sum()}  '
          f'covid={(val_df["label"]==1).sum()})')
    print(f'  Test : {len(test_df):,}  '
          f'(normal={(test_df["label"]==0).sum()}  '
          f'covid={(test_df["label"]==1).sum()})')

    # ── Build loaders ─────────────────────────────────────────────────────────
    train_loader, val_loader, test_loader = make_loaders(
        train_df, val_df, test_df, BATCH_SIZE
    )

    # ── Extract features once per cohort (frozen encoder = deterministic) ─────
    print('\nExtracting features...')
    train_data = extract_biomedclip_features(biomedclip_encoder, train_loader, DEVICE)
    val_data   = extract_biomedclip_features(biomedclip_encoder, val_loader,   DEVICE)
    test_data  = extract_biomedclip_features(biomedclip_encoder, test_loader,  DEVICE)

    print(f'  Train embeddings: {train_data["latents"].shape}')
    print(f'  Val   embeddings: {val_data["latents"].shape}')
    print(f'  Test  embeddings: {test_data["latents"].shape}')

    # ── Run probe across all 3 seeds ──────────────────────────────────────────
    cohort_results = []

    for seed in SEEDS:
        set_seed(seed)
        print(f'\n── {COHORT} | Seed {seed} ──')
        result = run_probe(train_data, val_data, test_data, COHORT, seed)
        cohort_results.append(result)
        all_results.append(result)

        print(f"  Val  AUC : {result['val_auc']:.4f}")
        print(f"  Test AUC : {result['test_auc']:.4f}  "
              f"(gap: {result['val_test_gap']:+.4f})")
        print(f"  Test Acc : {result['test_acc']:.4f}")
        print(f"  Test F1  : {result['test_f1']:.4f}")
        miss = result['miss_rate']
        flag = 'CRITICAL' if miss > 0.20 else 'OK'
        print(f"  Miss Rate: {miss:.4f}  ({flag})")

    # ── Per-cohort summary + save ─────────────────────────────────────────────
    cohort_df = pd.DataFrame(cohort_results)
    summarise(cohort_df)
    cohort_df.to_csv(f'biomedclip_results_{COHORT}.csv', index=False)
    print(f'\nSaved: biomedclip_results_{COHORT}.csv')

# ── Master summary ────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('MASTER SUMMARY — ALL COHORTS (mean ± std over 3 seeds)')
print(f'{"="*60}')

master_df = pd.DataFrame(all_results)

for cohort in COHORTS:
    sub = master_df[master_df['cohort'] == cohort]
    print(f"\n  {cohort}")
    print(f"    AUC      : {sub['test_auc'].mean():.4f} ± {sub['test_auc'].std():.4f}")
    print(f"    F1       : {sub['test_f1'].mean():.4f} ± {sub['test_f1'].std():.4f}")
    print(f"    Miss Rate: {sub['miss_rate'].mean():.4f} ± {sub['miss_rate'].std():.4f}")
    print(f"    Val-Test Gap: {sub['val_test_gap'].mean():+.4f} ± {sub['val_test_gap'].std():.4f}")

master_df.to_csv('biomedclip_results_ALL.csv', index=False)
print('\nAll results saved to: biomedclip_results_ALL.csv')


Loading full dataset CSVs...
  Train: 6,267 rows  |  Val: 1,354  |  Test: 1,363
  Severity counts in train: {0: 1910, 1: 1931, 2: 1798, 3: 628}

BiomedCLIP Baseline — Cohort S1 (severity 1 vs 0)
  Train: 3,841  (normal=1910  covid=1931)
  Val  : 832  (normal=423  covid=409)
  Test : 827  (normal=417  covid=410)

Extracting features...


Extracting BiomedCLIP features: 100%|██████████| 13/13 [00:10<00:00,  1.26it/s]


  Train embeddings: (3841, 512)
  Val   embeddings: (832, 512)
  Test  embeddings: (827, 512)

── S1 | Seed 16 ──
  Val  AUC : 0.7303
  Test AUC : 0.6280  (gap: +0.1023)
  Test Acc : 0.5865
  Test F1  : 0.5476
  Miss Rate: 0.4951  (CRITICAL)

── S1 | Seed 42 ──
  Val  AUC : 0.7303
  Test AUC : 0.6280  (gap: +0.1023)
  Test Acc : 0.5865
  Test F1  : 0.5476
  Miss Rate: 0.4951  (CRITICAL)

── S1 | Seed 999 ──
  Val  AUC : 0.7303
  Test AUC : 0.6280  (gap: +0.1023)
  Test Acc : 0.5865
  Test F1  : 0.5476
  Miss Rate: 0.4951  (CRITICAL)

BiomedCLIP — Cohort S1 (mean ± std, 3 seeds)
  val_acc           : 0.6599 ± 0.0000
  test_acc          : 0.5865 ± 0.0000
  test_f1           : 0.5476 ± 0.0000
  test_auc          : 0.6280 ± 0.0000
  miss_rate         : 0.4951 ± 0.0000
  val_test_gap      : 0.1023 ± 0.0000

Per-seed val–test AUC gap (shortcut learning indicator):
  Seed  16: val=0.7303  test=0.6280  gap=+0.1023
  Seed  42: val=0.7303  test=0.6280  gap=+0.1023
  Seed 999: val=0.7303  test=0.

Extracting BiomedCLIP features: 100%|██████████| 13/13 [00:09<00:00,  1.36it/s]


  Train embeddings: (3708, 512)
  Val   embeddings: (808, 512)
  Test  embeddings: (808, 512)

── S2 | Seed 16 ──
  Val  AUC : 0.8389
  Test AUC : 0.8624  (gap: -0.0235)
  Test Acc : 0.7661
  Test F1  : 0.7497
  Miss Rate: 0.2762  (CRITICAL)

── S2 | Seed 42 ──
  Val  AUC : 0.8389
  Test AUC : 0.8624  (gap: -0.0235)
  Test Acc : 0.7661
  Test F1  : 0.7497
  Miss Rate: 0.2762  (CRITICAL)

── S2 | Seed 999 ──
  Val  AUC : 0.8389
  Test AUC : 0.8624  (gap: -0.0235)
  Test Acc : 0.7661
  Test F1  : 0.7497
  Miss Rate: 0.2762  (CRITICAL)

BiomedCLIP — Cohort S2 (mean ± std, 3 seeds)
  val_acc           : 0.7574 ± 0.0000
  test_acc          : 0.7661 ± 0.0000
  test_f1           : 0.7497 ± 0.0000
  test_auc          : 0.8624 ± 0.0000
  miss_rate         : 0.2762 ± 0.0000
  val_test_gap      : -0.0235 ± 0.0000

Per-seed val–test AUC gap (shortcut learning indicator):
  Seed  16: val=0.8389  test=0.8624  gap=-0.0235
  Seed  42: val=0.8389  test=0.8624  gap=-0.0235
  Seed 999: val=0.8389  test=0

Extracting BiomedCLIP features: 100%|██████████| 9/9 [00:07<00:00,  1.24it/s]


  Train embeddings: (2538, 512)
  Val   embeddings: (560, 512)
  Test  embeddings: (562, 512)

── S3 | Seed 16 ──
  Val  AUC : 0.8278
  Test AUC : 0.9190  (gap: -0.0913)
  Test Acc : 0.8826
  Test F1  : 0.7273
  Miss Rate: 0.3931  (CRITICAL)

── S3 | Seed 42 ──
  Val  AUC : 0.8278
  Test AUC : 0.9190  (gap: -0.0912)
  Test Acc : 0.8826
  Test F1  : 0.7273
  Miss Rate: 0.3931  (CRITICAL)

── S3 | Seed 999 ──
  Val  AUC : 0.8278
  Test AUC : 0.9190  (gap: -0.0912)
  Test Acc : 0.8826
  Test F1  : 0.7273
  Miss Rate: 0.3931  (CRITICAL)

BiomedCLIP — Cohort S3 (mean ± std, 3 seeds)
  val_acc           : 0.8357 ± 0.0000
  test_acc          : 0.8826 ± 0.0000
  test_f1           : 0.7273 ± 0.0000
  test_auc          : 0.9190 ± 0.0000
  miss_rate         : 0.3931 ± 0.0000
  val_test_gap      : -0.0912 ± 0.0000

Per-seed val–test AUC gap (shortcut learning indicator):
  Seed  16: val=0.8278  test=0.9190  gap=-0.0913
  Seed  42: val=0.8278  test=0.9190  gap=-0.0912
  Seed 999: val=0.8278  test=0

## 10. Memory check (useful for Kaggle T4)

ViT-B/16 inference at batch=64 uses ~3GB VRAM.
If you hit OOM, reduce BATCH_SIZE to 32.

In [12]:
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated(DEVICE)  / 1024**3
    cached = torch.cuda.memory_reserved(DEVICE)   / 1024**3
    total  = torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3
    print(f'GPU memory — allocated: {alloc:.2f} GB  '
          f'cached: {cached:.2f} GB  '
          f'total: {total:.2f} GB')
    print(f'Headroom: {total - cached:.2f} GB')

GPU memory — allocated: 0.33 GB  cached: 0.96 GB  total: 14.56 GB
Headroom: 13.60 GB


---
## Notes for paper write-up

**Preprocessing:** CT slices (single-channel float32, range [-1,1]) were converted
to pseudo-RGB by channel repetition and renormalized to BiomedCLIP's expected
ImageNet statistics (mean=[0.481,0.457,0.408], std=[0.269,0.261,0.276]) before
resizing to 224×224 via bicubic interpolation. The BiomedCLIP vision encoder
(ViT-B/16) was used in frozen mode; no fine-tuning of encoder weights was
performed. A linear probe (Logistic Regression for S1/S2, RBF-SVM for S3)
was trained on L2-normalized 512-dimensional image embeddings using the same
volume-level train/val/test splits as PAR-VAE and the CNN baseline.

**What to look for in results:**
- If BiomedCLIP also hits ~67% AUC on S1 → the ceiling is confirmed
  as representation-independent (strengthens your biological ceiling claim)
- If BiomedCLIP shows large val–test gap at S3 → confirms shortcut learning
  is not specific to simple CNNs but persists in SOTA VLMs without physics grounding
- If BiomedCLIP has high miss rate at S3 → strongest possible safety argument
  for PAR-VAE's physics constraints